[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/17_multi_armed_bandits/notebooks/02_frecuentistas.ipynb)

# Algoritmos Frecuentistas para Bandidos Multibrazo

**Módulo 17 — Multi-Armed Bandits · Notebook 02**

---

## ¿De qué trata este notebook?

Implementamos y comparamos los tres algoritmos frecuentistas principales para el problema del bandido multibrazo:

1. **ε-Greedy** — la estrategia más simple: exploración aleatoria con probabilidad ε
2. **UCB1** — exploración dirigida por intervalos de confianza
3. **KL-UCB** — cotas más ajustadas usando la divergencia KL

Cada implementación sigue directamente el pseudocódigo de las Secciones 02 y 03.
Reproducimos las trazas manuales y comparamos el regret acumulado.

**Prerequisitos**: haber leído las Secciones 17.1–17.3 del módulo.

In [ ]:
# Solo en Colab — en entorno local estas librerías ya deben estar instaladas
import sys
if "google.colab" in sys.modules:
    !pip install -q numpy matplotlib scipy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

COLORS = {
    "red": "#E94F37", "orange": "#F28C28", "blue": "#2E86AB",
    "green": "#27AE60", "purple": "#8E44AD", "gray": "#7F8C8D",
}
ARM_COLORS = [COLORS["red"], COLORS["orange"], COLORS["blue"]]
ARM_NAMES = ["A", "B", "C"]

---

## Entorno: BanditEnvironment

Redefinimos el entorno para que este notebook sea **autocontenido**.
Soporta recompensas Bernoulli (Problema Canónico A) y Gaussianas (Problema Canónico B).

In [ ]:
class BanditEnvironment:
    """Multi-armed bandit environment.

    Parameters
    ----------
    means : list[float]
        True mean reward for each arm.
    reward_type : str
        'bernoulli' or 'gaussian'.
    sigma : float
        Std dev for Gaussian arms (ignored if Bernoulli).
    """

    def __init__(self, means, reward_type="bernoulli", sigma=1.0):
        self.means = np.array(means, dtype=float)
        self.K = len(means)
        self.reward_type = kind
        self.sigma = sigma
        self.mu_star = np.max(self.means)
        self.best_arm = int(np.argmax(self.means))

    def pull(self, arm):
        """Pull an arm and return a stochastic reward."""
        if self.reward_type == "bernoulli":
            return float(np.random.random() < self.means[arm])
        else:  # gaussian
            return np.random.normal(self.means[arm], self.sigma)

    def regret(self, arm):
        """Instantaneous regret of pulling `arm`."""
        return self.mu_star - self.means[arm]


# ── Problemas canónicos ────────────────────────────────────
CANONICAL_A = dict(means=[0.3, 0.5, 0.7], reward_type="bernoulli")
CANONICAL_B = dict(means=[1.0, 2.0, 3.0], reward_type="gaussian", sigma=1.5)

print("Problema Canónico A (Bernoulli):", CANONICAL_A)
print("Problema Canónico B (Gaussiano):", CANONICAL_B)

---

## 1. ε-Greedy

La idea es directa: con probabilidad $\varepsilon$ **explorar** (brazo aleatorio),
con probabilidad $1-\varepsilon$ **explotar** (argmax de las estimaciones).

La clase siguiente traduce directamente el pseudocódigo de la Sección 02.
Los comentarios `[P1]`–`[P5]` indican la correspondencia línea por línea.

In [ ]:
class EpsilonGreedy:
    """ε-Greedy algorithm for multi-armed bandits.

    Matches pseudocode labels [P1]–[P5] from Section 02.
    """

    def __init__(self, K, epsilon=0.1):
        self.K = K
        self.epsilon = epsilon
        self.Q = np.zeros(K)        # [P1] estimated mean reward per arm
        self.N = np.zeros(K, int)   # [P2] pull count per arm

    def select(self):
        """Choose an arm: explore with prob ε, exploit otherwise."""
        if np.random.random() < self.epsilon:              # [P3] explore
            return np.random.randint(self.K)
        else:                                               # [P4] exploit
            return int(np.argmax(self.Q))

    def update(self, arm, reward):
        """Incremental mean update."""
        self.N[arm] += 1
        self.Q[arm] += (reward - self.Q[arm]) / self.N[arm]  # [P5] incremental mean

    def __repr__(self):
        return f"EpsilonGreedy(ε={self.epsilon})"

---

### Ejecutar ε-Greedy en el Problema Canónico A

Corremos 1000 rondas con `seed=42` y visualizamos:
- **Selección de brazos** (scatter coloreado por brazo)
- **Regret acumulado**

In [ ]:
def run_agent(agent, env, T, seed=42):
    """Run an agent on an environment for T rounds.

    Returns
    -------
    arms : ndarray of int, shape (T,)
    rewards : ndarray of float, shape (T,)
    cum_regret : ndarray of float, shape (T,)
    """
    np.random.seed(seed)
    arms = np.empty(T, dtype=int)
    rewards = np.empty(T)
    regrets = np.empty(T)

    for t in range(T):
        a = agent.select()
        r = env.pull(a)
        agent.update(a, r)
        arms[t] = a
        rewards[t] = r
        regrets[t] = env.regret(a)

    return arms, rewards, np.cumsum(regrets)


def plot_run(arms, cum_regret, title=""):
    """Plot arm selection scatter + cumulative regret."""
    T = len(arms)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: arm selection scatter
    ax = axes[0]
    for k in range(3):
        mask = arms == k
        ax.scatter(np.where(mask)[0], arms[mask],
                   color=ARM_COLORS[k], s=4, alpha=0.5, label=f"Brazo {ARM_NAMES[k]}")
    ax.set_yticks([0, 1, 2])
    ax.set_yticklabels(ARM_NAMES)
    ax.set_xlabel("Ronda $t$")
    ax.set_ylabel("Brazo seleccionado")
    ax.set_title(f"Selección de brazos — {title}")
    ax.legend(markerscale=4)

    # Right: cumulative regret
    ax = axes[1]
    ax.plot(cum_regret, color=COLORS["blue"], linewidth=1.5)
    ax.set_xlabel("Ronda $t$")
    ax.set_ylabel("Regret acumulado $R_t$")
    ax.set_title(f"Regret acumulado — {title}")

    plt.tight_layout()
    plt.show()


# ── Run ε-greedy ──────────────────────────────────────────
env_a = BanditEnvironment(**CANONICAL_A)
agent_eg = EpsilonGreedy(K=3, epsilon=0.1)
arms_eg, rewards_eg, regret_eg = run_agent(agent_eg, env_a, T=1000, seed=42)

plot_run(arms_eg, regret_eg, title="ε-Greedy (ε=0.1)")

print(f"Regret final: {regret_eg[-1]:.2f}")
print(f"Pulls por brazo: A={agent_eg.N[0]}, B={agent_eg.N[1]}, C={agent_eg.N[2]}")

---

### Reproducir la traza manual (primeras 10 rondas)

Verificamos paso a paso que nuestra implementación coincide con la tabla de la Sección 02.
Cada fila muestra: ronda, tipo de acción (explore/exploit), brazo elegido, recompensa,
estimaciones actualizadas y regret acumulado.

In [ ]:
# ── Hand trace: first 10 steps ──────────────────────────
np.random.seed(42)
env_trace = BanditEnvironment(**CANONICAL_A)
agent_trace = EpsilonGreedy(K=3, epsilon=0.1)
cum_reg = 0.0

header = f"{'t':>3} | {'acción':^10} | {'At':^4} | {'rt':^4} | {'μ̂_A':>6} | {'μ̂_B':>6} | {'μ̂_C':>6} | {'N_A':>3} | {'N_B':>3} | {'N_C':>3} | {'Rt':>6}"
print(header)
print("-" * len(header))

for t in range(1, 11):
    # Peek at the random value to determine explore/exploit
    u = np.random.random()
    if u < agent_trace.epsilon:
        action = "explore"
        arm = np.random.randint(agent_trace.K)
    else:
        action = "exploit"
        arm = int(np.argmax(agent_trace.Q))

    reward = env_trace.pull(arm)
    agent_trace.update(arm, reward)
    cum_reg += env_trace.regret(arm)

    print(f"{t:3d} | {action:^10} | {ARM_NAMES[arm]:^4} | {reward:4.0f} | "
          f"{agent_trace.Q[0]:6.3f} | {agent_trace.Q[1]:6.3f} | {agent_trace.Q[2]:6.3f} | "
          f"{agent_trace.N[0]:3d} | {agent_trace.N[1]:3d} | {agent_trace.N[2]:3d} | "
          f"{cum_reg:6.1f}")

---

### Esquemas de decaimiento de ε

El ε constante nunca deja de explorar, lo que genera regret lineal.
Implementamos tres variantes:

| Variante | Fórmula | Propiedad |
|----------|---------|----------|
| Constante | $\varepsilon_t = \varepsilon_0$ | Explora para siempre |
| Decaimiento lineal | $\varepsilon_t = \varepsilon_0 (1 - t/T)$ | Requiere conocer $T$ |
| Decaimiento $1/t$ | $\varepsilon_t = c / (c + t)$ | Automático |

In [ ]:
class EpsilonGreedyDecay:
    """ε-Greedy with configurable decay schedule."""

    def __init__(self, K, epsilon0=0.5, schedule="constant", T=1000, c=10.0):
        self.K = K
        self.epsilon0 = epsilon0
        self.schedule = schedule
        self.T = T
        self.c = c
        self.Q = np.zeros(K)
        self.N = np.zeros(K, int)
        self.t = 0

    def _get_epsilon(self):
        if self.schedule == "constant":
            return self.epsilon0
        elif self.schedule == "linear":
            return self.epsilon0 * max(0, 1 - self.t / self.T)
        elif self.schedule == "inverse":
            return self.c / (self.c + self.t)
        else:
            raise ValueError(f"Unknown schedule: {self.schedule}")

    def select(self):
        eps = self._get_epsilon()
        if np.random.random() < eps:
            return np.random.randint(self.K)
        return int(np.argmax(self.Q))

    def update(self, arm, reward):
        self.t += 1
        self.N[arm] += 1
        self.Q[arm] += (reward - self.Q[arm]) / self.N[arm]

    def __repr__(self):
        return f"EpsilonGreedyDecay({self.schedule}, ε0={self.epsilon0})"


# ── Compare three schedules on Canonical A ──────────────
T = 1000
n_runs = 200
schedules = [
    ("constant",  {"epsilon0": 0.1, "schedule": "constant"}),
    ("linear",    {"epsilon0": 0.5, "schedule": "linear", "T": T}),
    ("1/t",       {"epsilon0": 1.0, "schedule": "inverse", "c": 10.0}),
]
schedule_colors = [COLORS["red"], COLORS["blue"], COLORS["green"]]

fig, ax = plt.subplots(figsize=(10, 6))

for (name, params), color in zip(schedules, schedule_colors):
    all_regret = np.zeros((n_runs, T))
    for run in range(n_runs):
        np.random.seed(run)
        env = BanditEnvironment(**CANONICAL_A)
        agent = EpsilonGreedyDecay(K=3, **params)
        _, _, cum_reg = run_agent(agent, env, T, seed=run)
        all_regret[run] = cum_reg

    mean_reg = all_regret.mean(axis=0)
    std_reg = all_regret.std(axis=0)
    ax.plot(mean_reg, label=name, color=color, linewidth=1.5)
    ax.fill_between(range(T), mean_reg - std_reg, mean_reg + std_reg,
                    alpha=0.15, color=color)

ax.set_xlabel("Ronda $t$")
ax.set_ylabel("Regret acumulado $R_t$")
ax.set_title("Regret: esquemas de decaimiento de ε (Canónico A)")
ax.legend()
plt.tight_layout()
plt.show()

**Ejercicio:** Implementa tu propio esquema de decaimiento (por ejemplo, exponencial
$\varepsilon_t = \varepsilon_0 \cdot \gamma^t$ con $\gamma = 0.995$).
Modifica la celda anterior para incluirlo en la comparación.

In [ ]:
# ── Tu esquema de decaimiento personalizado ──────────────

# Pista: copia EpsilonGreedyDecay y agrega un elif en _get_epsilon()
custom_schedule = "exponential"  # <-- CHANGE THIS
gamma = 0.995                     # <-- CHANGE THIS

# TODO: implementar y agregar al gráfico de comparación

---

## 2. UCB1: optimismo ante la incertidumbre

UCB1 resuelve el defecto fundamental de ε-greedy: en lugar de explorar **a ciegas**,
explora donde hay más **incertidumbre**.

Comparado con ε-greedy, solo hay **tres cambios** (marcados `[C1]`–`[C3]` en el código):

| Cambio | ε-Greedy | UCB1 |
|--------|----------|------|
| `[C1]` | Requiere parámetro ε | **Sin parámetros**: jala cada brazo una vez |
| `[C2]` | Coin flip aleatorio | **Bonus de confianza**: $\sqrt{2\ln t / N_i}$ |
| `[C3]` | Selección aleatoria (si explora) | **Determinista**: siempre argmax de UCB |

In [ ]:
class UCB1:
    """UCB1 algorithm for multi-armed bandits.

    Changes from ε-greedy marked [C1]–[C3].
    """

    def __init__(self, K):                              # [C1] no epsilon parameter
        self.K = K
        self.Q = np.zeros(K)
        self.N = np.zeros(K, int)
        self.t = 0

    def select(self):
        """Pull each arm once, then argmax of UCB values."""
        self.t += 1

        # [C1] initialization: pull each arm once
        for i in range(self.K):
            if self.N[i] == 0:
                return i

        # [C2] compute UCB values with confidence bonus
        bonus = np.sqrt(2 * np.log(self.t) / self.N)
        ucb_values = self.Q + bonus

        # [C3] deterministic argmax (no randomness)
        return int(np.argmax(ucb_values))

    def update(self, arm, reward):
        """Same incremental mean update as ε-greedy."""
        self.N[arm] += 1
        self.Q[arm] += (reward - self.Q[arm]) / self.N[arm]

    def __repr__(self):
        return "UCB1()"

---

### Ejecutar UCB1 y reproducir la traza manual

Corremos UCB1 con `seed=42` y verificamos las primeras 10 rondas
contra la tabla de la Sección 03. Mostramos además los valores UCB
de cada brazo para ver cómo el bonus guía la exploración.

In [ ]:
# ── Run UCB1 on Canonical A ──────────────────────────────
env_a = BanditEnvironment(**CANONICAL_A)
agent_ucb = UCB1(K=3)
arms_ucb, rewards_ucb, regret_ucb = run_agent(agent_ucb, env_a, T=1000, seed=42)

plot_run(arms_ucb, regret_ucb, title="UCB1")

print(f"Regret final: {regret_ucb[-1]:.2f}")
print(f"Pulls por brazo: A={agent_ucb.N[0]}, B={agent_ucb.N[1]}, C={agent_ucb.N[2]}")

In [ ]:
# ── Hand trace: UCB1 first 10 steps ─────────────────────
np.random.seed(42)
env_trace2 = BanditEnvironment(**CANONICAL_A)
agent_trace2 = UCB1(K=3)

header = (f"{'t':>3} | {'At':^4} | {'rt':^4} | "
          f"{'μ̂_A':>6} | {'c_A':>5} | {'UCB_A':>6} | "
          f"{'μ̂_B':>6} | {'c_B':>5} | {'UCB_B':>6} | "
          f"{'μ̂_C':>6} | {'c_C':>5} | {'UCB_C':>6}")
print(header)
print("-" * len(header))

for t in range(1, 11):
    arm = agent_trace2.select()
    reward = env_trace2.pull(arm)
    agent_trace2.update(arm, reward)

    # Compute current UCB values for display
    if np.all(agent_trace2.N > 0):
        bonus = np.sqrt(2 * np.log(agent_trace2.t) / agent_trace2.N)
        ucb_vals = agent_trace2.Q + bonus
        line = (f"{t:3d} | {ARM_NAMES[arm]:^4} | {reward:4.0f} | "
                f"{agent_trace2.Q[0]:6.3f} | {bonus[0]:5.2f} | {ucb_vals[0]:6.2f} | "
                f"{agent_trace2.Q[1]:6.3f} | {bonus[1]:5.2f} | {ucb_vals[1]:6.2f} | "
                f"{agent_trace2.Q[2]:6.3f} | {bonus[2]:5.2f} | {ucb_vals[2]:6.2f}")
    else:
        # During initialization phase, not all arms pulled yet
        qs = [f"{agent_trace2.Q[i]:6.3f}" if agent_trace2.N[i] > 0 else "     —" for i in range(3)]
        line = (f"{t:3d} | {ARM_NAMES[arm]:^4} | {reward:4.0f} | "
                f"{qs[0]} |    — |      — | "
                f"{qs[1]} |    — |      — | "
                f"{qs[2]} |    — |      —")
    print(line)

---

## 3. KL-UCB: cotas más ajustadas

UCB1 usa la desigualdad de Hoeffding, que es independiente de la distribución.
**KL-UCB** aprovecha que sabemos que las recompensas son Bernoulli para obtener
cotas más **ajustadas**, especialmente cuando $\hat{\mu}_i$ está cerca de 0 o 1.

En lugar de sumar un bonus aditivo, KL-UCB encuentra el mayor $q$ tal que:

$$\text{KL}(\hat{\mu}_i, q) \leq \frac{\ln t}{N_i(t)}$$

Usamos **búsqueda binaria** para encontrar $q$.

In [ ]:
def kl_bernoulli(p, q):
    """KL divergence between Bernoulli(p) and Bernoulli(q).

    KL(p, q) = p * ln(p/q) + (1-p) * ln((1-p)/(1-q))
    """
    # Clip to avoid log(0)
    p = np.clip(p, 1e-10, 1 - 1e-10)
    q = np.clip(q, 1e-10, 1 - 1e-10)
    return p * np.log(p / q) + (1 - p) * np.log((1 - p) / (1 - q))


def kl_ucb_bound(p_hat, threshold, n_iter=50):
    """Find max q in [p_hat, 1] s.t. KL(p_hat, q) <= threshold via binary search."""
    lo, hi = p_hat, 1.0 - 1e-10
    for _ in range(n_iter):
        mid = (lo + hi) / 2
        if kl_bernoulli(p_hat, mid) <= threshold:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2


class KLUCB:
    """KL-UCB algorithm for Bernoulli bandits."""

    def __init__(self, K):
        self.K = K
        self.Q = np.zeros(K)
        self.N = np.zeros(K, int)
        self.t = 0

    def select(self):
        self.t += 1
        # Pull each arm once
        for i in range(self.K):
            if self.N[i] == 0:
                return i

        # Compute KL-UCB index for each arm
        threshold = np.log(self.t)
        ucb_vals = np.array([
            kl_ucb_bound(self.Q[i], threshold / self.N[i])
            for i in range(self.K)
        ])
        return int(np.argmax(ucb_vals))

    def update(self, arm, reward):
        self.N[arm] += 1
        self.Q[arm] += (reward - self.Q[arm]) / self.N[arm]

    def __repr__(self):
        return "KLUCB()"

---

## 4. Comparación lado a lado en el Problema Canónico A

Corremos los tres algoritmos 200 veces y comparamos:
1. **Regret acumulado promedio** (con banda de ±1 desviación estándar)
2. **Distribución de pulls** por brazo

In [ ]:
T = 1000
n_runs = 200

algorithms = {
    "ε-Greedy (ε=0.1)": lambda: EpsilonGreedy(K=3, epsilon=0.1),
    "UCB1":              lambda: UCB1(K=3),
    "KL-UCB":            lambda: KLUCB(K=3),
}
algo_colors = [COLORS["red"], COLORS["blue"], COLORS["green"]]

results = {}
pull_counts = {}

for (name, make_agent), color in zip(algorithms.items(), algo_colors):
    all_regret = np.zeros((n_runs, T))
    all_pulls = np.zeros((n_runs, 3))

    for run in range(n_runs):
        env = BanditEnvironment(**CANONICAL_A)
        agent = make_agent()
        arms, _, cum_reg = run_agent(agent, env, T, seed=run)
        all_regret[run] = cum_reg
        for k in range(3):
            all_pulls[run, k] = np.sum(arms == k)

    results[name] = all_regret
    pull_counts[name] = all_pulls


# ── Plot 1: Cumulative regret comparison ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ax = axes[0]
for (name, reg), color in zip(results.items(), algo_colors):
    mean_r = reg.mean(axis=0)
    std_r = reg.std(axis=0)
    ax.plot(mean_r, label=name, color=color, linewidth=1.5)
    ax.fill_between(range(T), mean_r - std_r, mean_r + std_r,
                    alpha=0.12, color=color)

ax.set_xlabel("Ronda $t$")
ax.set_ylabel("Regret acumulado $R_t$")
ax.set_title("Regret acumulado promedio (200 runs, Canónico A)")
ax.legend()

# ── Plot 2: Pull distribution ────────────────────────────
ax = axes[1]
x = np.arange(3)
width = 0.25

for idx, (name, pulls) in enumerate(pull_counts.items()):
    mean_pulls = pulls.mean(axis=0)
    ax.bar(x + idx * width, mean_pulls, width, label=name,
           color=algo_colors[idx], alpha=0.8)

ax.set_xticks(x + width)
ax.set_xticklabels([f"Brazo {n}\n(μ={m})" for n, m in zip(ARM_NAMES, [0.3, 0.5, 0.7])])
ax.set_ylabel("Pulls promedio")
ax.set_title("Distribución de pulls (200 runs)")
ax.legend()

plt.tight_layout()
plt.show()

# ── Summary table ────────────────────────────────────────
print(f"{'Algoritmo':<20} | {'Regret final (μ ± σ)':>22} | {'Pulls A':>8} | {'Pulls B':>8} | {'Pulls C':>8}")
print("-" * 80)
for name in algorithms:
    final_reg = results[name][:, -1]
    mp = pull_counts[name].mean(axis=0)
    print(f"{name:<20} | {final_reg.mean():8.1f} ± {final_reg.std():6.1f}     | "
          f"{mp[0]:8.1f} | {mp[1]:8.1f} | {mp[2]:8.1f}")

---

## Ejercicio: Problema Canónico B (Gaussiano)

Repite la comparación anterior pero usando el **Problema Canónico B**
(Gaussiano con $\mu_A=1, \mu_B=2, \mu_C=3, \sigma=1.5$).

Nota: KL-UCB está diseñado para Bernoulli. Para Gaussianas,
usa solo ε-Greedy y UCB1.

**Preguntas:**
1. ¿Cómo cambia el regret respecto al caso Bernoulli?
2. ¿Qué algoritmo se beneficia más del cambio de distribución?

In [ ]:
# ── Tu código aquí: comparación en Canónico B ────────────

T_b = 1000       # <-- CHANGE THIS
n_runs_b = 200   # <-- CHANGE THIS

# Pista: reutiliza la misma estructura de la comparación anterior,
# pero con CANONICAL_B y sin KL-UCB.

# TODO: correr ε-Greedy y UCB1 en Canónico B
# TODO: graficar regret acumulado
# TODO: imprimir tabla resumen

---

## Reflexión

Responde estas preguntas antes de continuar al Notebook 03 (Thompson Sampling):

1. **¿Por qué UCB1 supera a ε-Greedy?**
   Piensa en qué información usa cada algoritmo al momento de decidir.
   ε-Greedy ignora cuántas veces ha jalado cada brazo; UCB1 lo incorpora
   directamente en el bonus $\sqrt{2\ln t / N_i}$.

2. **¿Qué significa "exploración dirigida" en la práctica?**
   Observa la gráfica de selección de brazos de UCB1 vs ε-Greedy.
   UCB1 concentra sus exploraciones al inicio (cuando la incertidumbre es alta)
   y después explota casi exclusivamente. ε-Greedy sigue explorando al azar
   incluso en $t=900$.

3. **¿Cuándo preferirías ε-Greedy sobre UCB1?**
   Piensa en escenarios donde las distribuciones de recompensa **cambian con el tiempo**
   (non-stationary bandits). ¿El bonus de UCB1 se adapta a eso?

4. **¿Por qué KL-UCB mejora sobre UCB1 en Bernoulli?**
   La cota de Hoeffding es universal pero conservadora.
   KL-UCB usa la estructura de la distribución Bernoulli para dar cotas más ajustadas,
   especialmente cuando $\hat{\mu}_i$ está cerca de 0 o 1.